## 2. Prompt Engineering & Inference
1. 怎么把问题喂给 LLM
2. 模型拿到概率分布后，怎么生成最终文本 

Inference: 
   - greedy methods (Fixed) : usually the model output the disctribution and always output the one with largest probability, which is not flexible. 
   code: torch.argmax(prob).item()

   - Beam search ( multiple path k ) : each next tokens will keep multiple k candidates . At last to evaluate the whole sentence, keep the best top k sentense , which would be higher computational cost. 

    import math

    # 每一步的候选概率（教学示例）
    steps = [
        {"A": 0.6, "B": 0.4},
        {"x": 0.5, "y": 0.5},
    ]

    beam_size = 2
    beams = [("", 0.0)]  # (sequence, log_prob)

    for step in steps:
        candidates = []
        for seq, score in beams:
            for token, prob in step.items():
                new_seq = seq + token
                new_score = score + math.log(prob)
                candidates.append((new_seq, new_score))
        candidates.sort(key=lambda x: x[1], reverse=True)
        beams = candidates[:beam_size]

    print(beams)
    


   - Sampling methods (do_sample = True): generate the random number based on the output distribution. 
         - code : torch.multinomial( probs ,num_samples = 1).item( )

         - top-k: only sample at the top-k words  (normalized then sample )

            - code: topk_probs, topk_indices = torch.topk(probs, k) 
                    topk_probs = topk_probs/topk_probs.sum() 
                    sampled_index = torch.multinomial(topk_probs, num_samples= 1) .item()
                    next_token = topk_indices[smapled_index].item() #  .item() get the Python 数值  from the tensor 

         - top-p : only sample the words from the accumulative probability top p 
            - code: sorted_probs, sorted_indices = torch.sort(probs, descending=True) 
                    cummulative = torch.cumsum( sorted_probs, dim = 0)
                    mask = cummulative <= p 

                    mask[0] = True 

                    filtered_probs = sorted_probs[ mask ]
                    filtered_indices = sorted_indices[ mask] 
                    filtered_probs = filtered_probs/ filtered_probs.sum()

                    smapled_index = torch.multinomial( filtered_probs, num_samples = 1) .items()
                    next_token = fitlered_indices[ sampled_index].item() 




#### Zero-shot / Few-shot prompting

- zero-shot: directly output from LLM 
    - template: You are a professional translator. Translate the following English sentence into natural Chinese: I love machine learning. 

- Few-shot: with examples in the prompt called "in-context learning" 
    - template: You are a professional translator. Translate the following English sentence into natural Chinese: I love machine learning.  for example: 
    { "I love NLP" : 我爱自然语言处理 }

    - Benefits: stable, output style is fixed, better for complex task 
    - Disadv: the prompt is longer, token wasted.  Wrong examples lead to bad result 

---- 
#### Chain-of-Thought (CoT): 让模型把中间推理步骤写出来，而不是只给最终答案, Thought thought answer
- With the sentence "Think step by step" in the Prompt 
    - usually works for the logic question, mathematics, science problem 

    - Disadantage: Can't update to the latest news, may hallucination 

- Few-shot CoT : With some step-by-step thinking examples in the prompt. **The questioner have to decode the problem into a few steps** 
    -exp: 
    prompt = f""" Q: If Tom has 3 apples and buys 2 more, how many does he have?
        A: Let's think step by step. Tom starts with 3 apples. He buys 2 more. 3 + 2 = 5. So the answer is 5.

        Q: If Sara has 10 candies and gives away 4, how many are left?
        A: Let's think step by step. Sara starts with 10 candies. She gives away 4. 10 - 4 = 6. So the answer is 6.

        Q: If Roger has 5 tennis balls and buys 2 cans with 3 each, how many does he have?
        A: Let's think step by step.""" 


- React 
    -  Thinking in text  + Interact with tools / environment  = Thought  act observation thought answer 

    - Exp : """ Question: What is the population of France?
            Thought: I should search for France population
            Action: search("France population") 
            Observation: 67 million
            Thought: I got the answer
            Answer: 67 million """ 
    



----
#### Prompt templates

- template:  with predefined variable article = "... "
 template = f"""
        You are a helpful assistant. 

        Task: Summarize the following article 

        Article: {article} 

        Answer : 
        """
- contains with examples 

    角色 You are a helpful assistant.  
    任务说明 Task: Summarize the following article  
    输入内容:  Article: {article}  and {context}, {question}, {prompt}... 
    输出格式要求: few short or defined strcutured output or Return JSON with keys: label, reason 
    限制条件 : Use only the provided context. 

---- 
#### Prompt tuning (train a prefix parameters connected with the question into the model)

    - Prompt tunning (prompt embedding): soft prompt : add the tokens before the input , [ soft prompt + input], the soft prompt can be tunned when training 

    - Prefix tunning (作用到每一层 attention 的 key/value): Add prefix for each attention layer 

    - p-Tunning  and p-Tunning v2 （ 深层次的prefix/prompt tuning） 




----
#### Temperature and decoding strategies, config 

- softmax(logits/T).  
    - T is larger, the low probability is closed to the high, thus higher probability to be choosem 
    - T = 1, the orignial distribution for the logits 
    - T< 1, the tokens with higher probability is more perfered .
---




## LLM template usually used in the examples 


- API
    - Gemini free 



- from huggingface, transformers 

    -Quantization mdoel 

    - Not quantization mdoel : 



- 

  next_token = topk_indices[smapled_index].item() why have the items 
 

In [4]:
import torch 
import math 

In [ ]:
steps = [
    {"A": 0.6, "B": 0.4},
    {"x": 0.5, "y": 0.5},
]

beam_size = 2
beams = [("", 0.0)]  # (sequence, log_prob)

for step in steps:
    candidates = []
    for seq, score in beams:
        for token, prob in step.items():
            new_seq = seq + token
            new_score = score + math.log(prob)
            candidates.append((new_seq, new_score))

    candidates.sort(key=lambda x: x[1], reverse=True)

    beams = candidates[:beam_size]


print(beams)
    

[('Ax', -1.203972804325936), ('Ay', -1.203972804325936)]


## React code 

Example with  a mini ReAct for quesiton 
"What is the square root of the population of France?"  

In [ ]:
while True: 
    Thought = Model( "what should i do")
    if "Action" in the Thought: 
        result = run_tool(...) 
        observation = result 
        
    else:
        break 

## Prompt tunning 
1. 在输入序列前加入若干可训练响亮， embedding space 里的参数 
2. Training to update 这些prompt vectors 
x′=[p1​,p2​,...,pm​,x1​,x2​,...,xn​] 

code 1: from cratch 
code2 : from huggingface  peft 


In [ ]:
import torch 
import torch.nn as nn 

class PromptTunning(nn.Module): 
    def __init__(): 
        super().__init__( base_embedding, prompt_length )
        self.embedding = base_embedding
        d_model = self.embedding.embedding_dim
        
        self.soft_prompt = nn.Parameters( torch.randn(prompt_length, d_model))
        
    def forward( self, input_ids): 
        """
        input_ids = [ batch, seq_len]
        """
        
        batch_size= input_ids.size[0]
        
        input_embeds= self.embedding(input_ids)# (seq_len, d_model )
        
        # prompt embeing (prompt_len, d_model )
        prompt_embeds = self.soft_prompt.unsqueeze(0).expand(batch_size, -1, -1)
        
        
        full_embeds = input_embeds + prompt_embeds
        
        return full_embeds 
        
        
    

In [ ]:
## Hugging face with peft 

from transformers import AutoModelForCausalLM, AutoTokenizer 
from peft import PromptTuningConfig , get_peft_model, TaskType 

model_promt = 'gpt2' 

tokenizer  = AutoTokenizer.from_pretrained(model_prompt)
tokenizer.pad_token = tokenizer.eos_token 

model = AutoModelForCasualLM.from_pretrained(model_prompt) 

# building the config for the soft promt tunning 
config = PromptTunningConfig( 
        task_type = TaskType.Casual_LM, 
        num_virtual_tokens = 20, # the number of prompt_length for the soft promt              
                )


# with the pretrained model and the config for the promt tunning 
# ( task_type, and the num_virtual _tokens )


peft_model = get_peft_model( model, config) 

peft_model.print_trainable_parameters() 


# test an example 
inputs = tokenizer( " The movie is good", return_tensors = 'pt')
output = model( **inputs, laebls = inputs['inputs_ids'])
print( output.loss)




## Prefix tunning  K/V 前面额外拼了一段可训练 prefix K/V , Q keep the same 
seq2seq / generation 任务 


In [ ]:
class SimpleSelfAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, prefix_k=None, prefix_v=None):
        """
        x: [B, L, H]
        prefix_k: [B, P, H] or None
        prefix_v: [B, P, H] or None
        """
        Q = self.q_proj(x)   # [B, L, H]
        K = self.k_proj(x)   # [B, L, H]
        V = self.v_proj(x)   # [B, L, H]

        # 如果有 prefix，就拼到 K/V 前面
        if prefix_k is not None and prefix_v is not None:
            K = torch.cat([prefix_k, K], dim=1)   # [B, P+L, H]
            V = torch.cat([prefix_v, V], dim=1)   # [B, P+L, H]

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.hidden_dim)
        attn_weights = F.softmax(scores, dim=-1)
        out = torch.matmul(attn_weights, V)       # [B, L, H]

        return self.out_proj(out) 
    



class SimpleTransformerBlock(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = SimpleSelfAttention(hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)

        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.ReLU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x, prefix_k=None, prefix_v=None):
        attn_out = self.attn(x, prefix_k=prefix_k, prefix_v=prefix_v)
        x = self.norm1(x + attn_out)

        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x 
    
    
class PrefixEncoder(nn.Module):
    
    def __init__(self, ): 
        super().__init__()
        
        self.num_layers = num_layer
        self.prefix_len = prefix_len 
        self.hidden_dim = hidden_dim 
        
        self.prefix_key = nn.Parameter( 
                        torch.randn(num_layers, prefix_len, hidden_dim ))

        self.prefix_value = nn.Parameter( 
                        torch.randn(num_layers, prefix_len, hidden_dim ))


    def forward(self, batch_size) : 
        # need to expand to the batch_size 
        prefix_k = self.prefix_key.unsqueeze(1).expand(-1, batch_size, -1, -1 )
        
        prefix_v = self.prefix_value.unsqueeze(1).expand(-1, batch_size, -1, -1 ) 
        
        
        return prefix_k, prefix_v 
    
     

In [ ]:
class PrefixTuningModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers, prefix_len, max_len=128):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embed = nn.Embedding(max_len, hidden_dim)

        self.blocks = nn.ModuleList([
            SimpleTransformerBlock(hidden_dim) for _ in range(num_layers)
        ])

        self.prefix_encoder = PrefixEncoder(
            num_layers=num_layers,
            prefix_len=prefix_len,
            hidden_dim=hidden_dim
        )

        self.lm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        """
        input_ids: [B, L]
        """
        B, L = input_ids.shape
        device = input_ids.device

        pos_ids = torch.arange(L, device=device).unsqueeze(0).expand(B, L)
        x = self.token_embed(input_ids) + self.pos_embed(pos_ids)   # [B, L, H]

        all_prefix_k, all_prefix_v = self.prefix_encoder(B)

        for layer_idx, block in enumerate(self.blocks):
            prefix_k = all_prefix_k[layer_idx]   # [B, P, H]
            prefix_v = all_prefix_v[layer_idx]   # [B, P, H]
            x = block(x, prefix_k=prefix_k, prefix_v=prefix_v)

        logits = self.lm_head(x)   # [B, L, vocab_size]
        return logits 
    
    

In [ ]:
from transformers import AutoTokenForSeq2SeqLM 

from peft import PrefixTuningConfig

model_prefix = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_prefix )
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config = PrefixTuningConfig( 
            task_type = TaskType.SEQ_2_SEQ_LM, 
            num_virtual_tokens = 20 )


model =get_peft_model( model , config) 
print(model.print_trainable_parameters()) 


 

## P-tunning 
先定义一些 pseudo tokens
再用一个 prompt encoder（比如 LSTM / MLP）  to learn the soft prompt 

## P-tuning v2
说对每层都加 prompt 

In [ ]:
class PromptEncoder(nn.Module):
    def __init__(self, prompt_len, hidden_dim):
        super().__init__()
        self.prompt_len = prompt_len
        self.hidden_dim = hidden_dim

        # pseudo token ids 对应的 embedding
        self.prompt_embedding = nn.Embedding(prompt_len, hidden_dim)

        # 用一个 BiLSTM + MLP 生成更强的 prompt 表示
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, batch_size, device):
        """
        返回 soft prompt embeddings: [B, P, H]
        """
        prompt_ids = torch.arange(self.prompt_len, device=device).unsqueeze(0)
        prompt_ids = prompt_ids.expand(batch_size, -1)  # [B, P]

        x = self.prompt_embedding(prompt_ids)  # [B, P, H]
        x, _ = self.lstm(x)
        x = self.mlp(x)
        return x
    
    
class PTuningModel(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers, prompt_len, max_len=128):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embed = nn.Embedding(max_len + prompt_len, hidden_dim)

        self.prompt_encoder = PromptEncoder(prompt_len, hidden_dim)

        self.blocks = nn.ModuleList([
            SimpleTransformerBlock(hidden_dim) for _ in range(num_layers)
        ])

        self.lm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        """
        input_ids: [B, L]
        """
        B, L = input_ids.shape
        device = input_ids.device

        # 原始 token embedding
        token_embeds = self.token_embed(input_ids)   # [B, L, H]

        # P-Tuning 生成的 soft prompt
        prompt_embeds = self.prompt_encoder(B, device)  # [B, P, H]
        P = prompt_embeds.size(1)

        # 拼到输入前面
        x = torch.cat([prompt_embeds, token_embeds], dim=1)   # [B, P+L, H]

        pos_ids = torch.arange(P + L, device=device).unsqueeze(0).expand(B, P + L)
        x = x + self.pos_embed(pos_ids)

        for block in self.blocks:
            x = block(x)

        logits = self.lm_head(x)   # [B, P+L, vocab_size]
        return logits 

## p-tunning v2 

In [ ]:
 ## 
class DeepPromptEncoder(nn.Module):
        def __init__(self, num_layers, prompt_len, hidden_dim):
        super().__init__()
        self.deep_prompts = nn.Parameter(
            torch.randn(num_layers, prompt_len, hidden_dim)
        )

    def forward(self, batch_size):
        """
        返回:
        [num_layers, B, P, H]
        """
        return self.deep_prompts.unsqueeze(1).expand(-1, batch_size, -1, -1)
     
     
class PTuningV2Model(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers, prompt_len, max_len=128):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embed = nn.Embedding(max_len + prompt_len, hidden_dim)

        self.blocks = nn.ModuleList([
            SimpleTransformerBlock(hidden_dim) for _ in range(num_layers)
        ])

        self.deep_prompt_encoder = DeepPromptEncoder(
            num_layers=num_layers,
            prompt_len=prompt_len,
            hidden_dim=hidden_dim
        )

        self.lm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        """
        input_ids: [B, L]
        """
        B, L = input_ids.shape
        device = input_ids.device

        pos_ids = torch.arange(L, device=device).unsqueeze(0).expand(B, L)
        x = self.token_embed(input_ids) + self.pos_embed(pos_ids)   # [B, L, H]

        deep_prompts = self.deep_prompt_encoder(B)   # [num_layers, B, P, H]

        for layer_idx, block in enumerate(self.blocks):
            prompt = deep_prompts[layer_idx]   # [B, P, H]
            P = prompt.size(1)

            # 每层前都拼一个 deep prompt
            x_with_prompt = torch.cat([prompt, x], dim=1)  # [B, P+L, H]

            pos_ids = torch.arange(P + x.size(1), device=device).unsqueeze(0).expand(B, P + x.size(1))
            x_with_prompt = x_with_prompt + self.pos_embed(pos_ids)

            x_with_prompt = block(x_with_prompt)

            # 只保留真实 token 部分，丢掉 prompt 部分
            x = x_with_prompt[:, P:, :]

        logits = self.lm_head(x)
        return logits 

2 Adapter 类
Adapter Tuning
在每层插入小模块（adapter）
训练这些小模块 


- Lora approach:  不是加 prompt，而是对线性层权重更新写成低秩形式， inference will merge to the original weights. 
1. 容易和量化结合，形成 QLoRA 
2. 中小任务上更优秀  
but 要改模型层（如 q_proj, v_proj 等），不是单纯前面加 token 



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],   # gpt2 常见 target module 示例
)

peft_model = get_peft_model(model, config)
peft_model.print_trainable_parameters() 